# Phase 5: protect the test set, then explore training data

This phase has two jobs. First, reserve an exact holdout of whole matches using only match IDs and shot counts. Then conduct descriptive EDA on the training matches. The holdout stays sealed: do not inspect its goal labels, xG, or feature relationships while choosing features or analysis decisions.

The dataset contains 9,168 shots in 380 matches. To satisfy the assignment's 80/20 row target, the holdout contains exactly 1,834 shots. The fixed match-count allocation also contains 76 whole matches; the remaining 7,334 shots in 304 matches form training. Match counts alone determine the groups. No label-based optimization is used.

A deterministic seed and a subset-sum allocation find a fixed-size match group with the exact row count. `data/processed/split_membership.csv` and `docs/split_manifest.json` lock that decision to the Phase 4 shot CSV hash. Later phases must reuse this membership rather than creating another holdout.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

starting_directory = Path.cwd().resolve()
search_locations = (starting_directory, *starting_directory.parents)
project_root = next(
    (
        folder for folder in search_locations
        if (folder / ".venv").is_dir()
        and (folder / "README.md").is_file()
        and (folder / "scripts" / "reserve_holdout.py").is_file()
        and (folder / "scripts" / "explore_shots.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open the project folder in VS Code before running this notebook.")
expected_python = (project_root / ".venv").resolve()
if Path(sys.prefix).resolve() != expected_python:
    raise RuntimeError(f"Select the project .venv kernel. Expected {expected_python}; got {Path(sys.prefix).resolve()}")

print(f"Project folder: {project_root}")
print(f"Python interpreter: {sys.executable}")

## Reserve the holdout before reading goal outcomes

Run the reservation script first. It checks the Phase 4 output hash and reads only `match_id` to count shots per match. If a valid lock already exists, it verifies and reuses that split. It stops if the source or membership file changed; it does not silently assign different matches.

In [ ]:
reserve_script = project_root / "scripts" / "reserve_holdout.py"
reservation = subprocess.run(
    [sys.executable, str(reserve_script), "--root", str(project_root)],
    check=True,
    capture_output=True,
    text=True,
)
print(reservation.stdout)
if reservation.stderr:
    print(reservation.stderr)

## Review the match-only partition

This summary contains counts only. It does not show goal outcomes, xG values, or feature relationships for the holdout. Check that the exact target has been reached and that every match appears in one partition.

In [ ]:
import pandas as pd
from IPython.display import display

split_manifest = json.loads((project_root / "docs" / "split_manifest.json").read_text(encoding="utf-8"))
membership = pd.read_csv(project_root / "data" / "processed" / "split_membership.csv", dtype={"match_id": "int64"})
assert len(membership) == split_manifest["total_match_count"]
assert membership["match_id"].is_unique
assert set(membership["split"]) == {"train", "test"}
assert split_manifest["test_shot_count"] == 1834 and split_manifest["test_match_count"] == 76
assert split_manifest["train_shot_count"] == 7334 and split_manifest["train_match_count"] == 304
partition_summary = pd.DataFrame([
    {"partition": "training", "matches": split_manifest["train_match_count"], "shots": split_manifest["train_shot_count"]},
    {"partition": "held-out test (sealed)", "matches": split_manifest["test_match_count"], "shots": split_manifest["test_shot_count"]},
])
display(partition_summary)
print("Split algorithm:", split_manifest["algorithm"])
print("Seed:", split_manifest["seed"])
print("Source CSV SHA-256:", split_manifest["shot_csv_sha256"])
print("Membership SHA-256:", split_manifest["membership_csv_sha256"])

## Explore the training partition only

Now run the EDA script. It verifies the shot source and membership hashes, filters to the training match IDs, and computes every table and figure from training rows only. It does not examine held-out outcomes or xG. If the input hash changed, return to the prior phase and resolve the mismatch.

In [ ]:
eda_script = project_root / "scripts" / "explore_shots.py"
eda_run = subprocess.run(
    [sys.executable, str(eda_script), "--root", str(project_root)],
    check=True,
    capture_output=True,
    text=True,
)
print(eda_run.stdout)
if eda_run.stderr:
    print(eda_run.stderr)

## Read the compact findings and numeric tables

The report stores hashes and provenance for reproducibility. The view below shows a short training-only summary instead of expanding the full source-file hash inventory. Goal rates use the training labels; the no-goal majority accuracy is only a class-imbalance reference, not a trained model.

In [ ]:
eda_report = json.loads((project_root / "docs" / "phase_05_eda_report.json").read_text(encoding="utf-8"))
tables_dir = project_root / "reports" / "tables"
class_balance = pd.read_csv(tables_dir / "phase05_class_balance.csv")
categorical_rates = pd.read_csv(tables_dir / "phase05_categorical_goal_rates.csv")
context_rates = pd.read_csv(tables_dir / "phase05_context_flag_goal_rates.csv")
geometry_rates = pd.read_csv(tables_dir / "phase05_geometry_goal_rates.csv")

summary_fields = {
    "status": eda_report["status"],
    "training matches": eda_report["training_match_count"],
    "training shots": eda_report["training_shot_count"],
    "training goals": eda_report["training_goal_count"],
    "training non-goals": eda_report["training_non_goal_count"],
    "training goal rate": f"{eda_report['training_overall_goal_rate']:.1%}",
    "descriptive no-goal baseline accuracy": f"{eda_report['training_no_goal_majority_baseline_accuracy']:.1%}",
    "training penalties": eda_report["training_penalty_shot_count"],
    "training penalty goals": eda_report["training_penalty_goal_count"],
}
display(pd.Series(summary_fields, name="value").to_frame())
print("Class balance:")
display(class_balance)
print("Categories with fewer than 30 training shots (rates are unstable):")
display(categorical_rates.loc[categorical_rates["small_group_n_lt_30"]])
print("Training-only context flag rates:")
display(context_rates)
print("Training-only geometry band rates:")
display(geometry_rates)

### 1. Training class balance

The figure shows the outcome counts and percentages in training data. The always-non-goal accuracy is a descriptive baseline only; it is not evidence that a classifier learned anything.

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(project_root / "reports" / "figures" / "phase05_01_class_balance.png")))

### 2. Training shot map

StatsBomb locations use the documented 120-by-80 coordinate frame. The attacked goal is at x = 120, so the plot preserves the source direction and shows the attacking half. Distances are in native coordinate units, not metres.

In [ ]:
display(Image(filename=str(project_root / "reports" / "figures" / "phase05_02_training_shot_map.png")))

### 3. Shot geometry distributions

These two distributions describe where training shots occurred. They summarize location geometry only; they do not use or display the held-out partition.

In [ ]:
display(Image(filename=str(project_root / "reports" / "figures" / "phase05_03_geometry_distributions.png")))

### 4. Goal rate across geometry bands

Each point is the observed training goal rate in a fixed distance or angle interval. Error bars are 95% Wilson intervals under an independent-shot approximation. Shots within a match may be related, so the intervals can be too narrow.

In [ ]:
display(Image(filename=str(project_root / "reports" / "figures" / "phase05_04_goal_rate_by_geometry.png")))

### 5. Goal rate by body part and shot type

The chart compares training-only category rates. Small groups are flagged in the numeric table; do not overinterpret their percentages.

In [ ]:
display(Image(filename=str(project_root / "reports" / "figures" / "phase05_05_goal_rate_by_category.png")))

### 6. Goal rate by context flags

NOT FLAGGED means the provider did not record the positive flag. It does not establish that pressure, a first-time attempt, or an open goal was objectively absent. These are descriptive associations, not causal effects.

In [ ]:
display(Image(filename=str(project_root / "reports" / "figures" / "phase05_06_goal_rate_by_context_flags.png")))

## Checkpoint before modeling

Save the notebook with outputs. Read `docs/phase_05_eda_findings.md` and the full `docs/phase_05_eda_report.json`; verify that figures and tables are training-only and that the split/source hashes match their manifests. Do not unseal the test partition during Phase 6 preprocessing or model selection. Phase 6 must reuse this locked membership for training, grouped cross-validation, and the final held-out evaluation.